# Installation

In [1]:
# conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
# pip install transformers datasets sentencepiece lightning scikit-learn evaluate pandas numpy tqdm matplotlib

In [2]:
import torch
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

In [3]:
print(torch.__version__)
print('GPU:', torch.cuda.is_available())

2.5.1
GPU: True


# EDA

In [4]:
train_df = pd.read_csv("data/train.csv")
train_df.head()

,id,comment,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200
0,0,ทำไมปล่อยให้จุดพลุกันสนั่นหวั่นไหว,0,0,0,0,0,0,0,0,0,0,0,0
1,1,แจ้งว่าการจุดพลุต้องขออนุญาต ทำไมจุดกันมากมายข...,0,0,0,0,0,0,0,0,0,0,0,0
2,2,คาดว่ามีการจุดพลุไม่ขอทางกรุงเทพให้ถูกต้อง ส่ง...,0,0,0,0,0,0,0,0,0,0,0,0
3,3,ไม่แน่ใจ กทม อนุญาตให้ร้านชอคโกแลตวิลจุพลุถึงก...,0,0,0,0,0,0,0,0,0,0,0,0
4,4,ไม่ทราบใครจัดงานปีใหม่ละแวกนี้ เปิดเสียงเพลงดั...,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306419 entries, 0 to 306418
Data columns (total 14 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   id                                     306419 non-null  int64 
 1   comment                                304284 non-null  object
 2   สำนักงานตำรวจแห่งชาติ                  306419 non-null  int64 
 3   การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย      306419 non-null  int64 
 4   สภาเด็กและเยาวชนกรุงเทพมหานคร          306419 non-null  int64 
 5   กรมควบคุมมลพิษ                         306419 non-null  int64 
 6   กรมสรรพสามิต                           306419 non-null  int64 
 7   การไฟฟ้านครหลวง                        306419 non-null  int64 
 8   กรมทางหลวง                             306419 non-null  int64 
 9   สำนักงานประกันสุขภาพแห่งชาติ           306419 non-null  int64 
 10  การประปานครหลวง                        306419 non-null  int64 
 11  

In [6]:
train_df.isnull().sum()

id                                          0
comment                                  2135
สำนักงานตำรวจแห่งชาติ                       0
การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย           0
สภาเด็กและเยาวชนกรุงเทพมหานคร               0
กรมควบคุมมลพิษ                              0
กรมสรรพสามิต                                0
การไฟฟ้านครหลวง                             0
กรมทางหลวง                                  0
สำนักงานประกันสุขภาพแห่งชาติ                0
การประปานครหลวง                             0
คณะกรรมการการพัฒนาเศรษฐกิจ                  0
กระทรวงการท่องเที่ยวและกีฬา                 0
สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200       0
dtype: int64

In [7]:
train_df = train_df.dropna(subset=['comment'])
train_df.isnull().sum()

id                                       0
comment                                  0
สำนักงานตำรวจแห่งชาติ                    0
การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย        0
สภาเด็กและเยาวชนกรุงเทพมหานคร            0
กรมควบคุมมลพิษ                           0
กรมสรรพสามิต                             0
การไฟฟ้านครหลวง                          0
กรมทางหลวง                               0
สำนักงานประกันสุขภาพแห่งชาติ             0
การประปานครหลวง                          0
คณะกรรมการการพัฒนาเศรษฐกิจ               0
กระทรวงการท่องเที่ยวและกีฬา              0
สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200    0
dtype: int64

In [8]:
class_cols = train_df.columns[2:]
class_cols

Index(['สำนักงานตำรวจแห่งชาติ', 'การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย',
       'สภาเด็กและเยาวชนกรุงเทพมหานคร', 'กรมควบคุมมลพิษ', 'กรมสรรพสามิต',
       'การไฟฟ้านครหลวง', 'กรมทางหลวง', 'สำนักงานประกันสุขภาพแห่งชาติ',
       'การประปานครหลวง', 'คณะกรรมการการพัฒนาเศรษฐกิจ',
       'กระทรวงการท่องเที่ยวและกีฬา', 'สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200'],
      dtype='object')

In [9]:
label_counts = train_df[class_cols].sum(axis=1) 
# สมมติว่า row #5 = [0, 0, 1, 0, 1, 0, 0, 0,...] -> label_counts[5] = 2

print("=== Before ===")
for n in range(label_counts.max()+1):
    print(f"จำนวน comment ที่มี {n} labels: {(label_counts == n).sum()} คิดเป็น {(label_counts == n).mean()*100:.2f}%")

# เอากลุ่มที่มีไม่ถึง 10 samples ออก
rare_groups = label_counts.value_counts()[label_counts.value_counts() < 10].index
train_df = train_df[~label_counts.isin(rare_groups)].reset_index(drop=True)

label_counts = train_df[class_cols].sum(axis=1)

print("=== After ===")
for n in range(label_counts.max()+1):
    print(f"จำนวน comment ที่มี {n} labels: {(label_counts == n).sum()} คิดเป็น {(label_counts == n).mean()*100:.2f}%")

train_df['label_group'] = label_counts.astype(int)
train_df.head()

=== Before ===
จำนวน comment ที่มี 0 labels: 231285 คิดเป็น 76.01%
จำนวน comment ที่มี 1 labels: 70827 คิดเป็น 23.28%
จำนวน comment ที่มี 2 labels: 2116 คิดเป็น 0.70%
จำนวน comment ที่มี 3 labels: 54 คิดเป็น 0.02%
จำนวน comment ที่มี 4 labels: 2 คิดเป็น 0.00%
=== After ===
จำนวน comment ที่มี 0 labels: 231285 คิดเป็น 76.01%
จำนวน comment ที่มี 1 labels: 70827 คิดเป็น 23.28%
จำนวน comment ที่มี 2 labels: 2116 คิดเป็น 0.70%
จำนวน comment ที่มี 3 labels: 54 คิดเป็น 0.02%


,id,comment,สำนักงานตำรวจแห่งชาติ,การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย,สภาเด็กและเยาวชนกรุงเทพมหานคร,กรมควบคุมมลพิษ,กรมสรรพสามิต,การไฟฟ้านครหลวง,กรมทางหลวง,สำนักงานประกันสุขภาพแห่งชาติ,การประปานครหลวง,คณะกรรมการการพัฒนาเศรษฐกิจ,กระทรวงการท่องเที่ยวและกีฬา,สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200,label_group
0,0,ทำไมปล่อยให้จุดพลุกันสนั่นหวั่นไหว,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,แจ้งว่าการจุดพลุต้องขออนุญาต ทำไมจุดกันมากมายข...,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,คาดว่ามีการจุดพลุไม่ขอทางกรุงเทพให้ถูกต้อง ส่ง...,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,ไม่แน่ใจ กทม อนุญาตให้ร้านชอคโกแลตวิลจุพลุถึงก...,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,ไม่ทราบใครจัดงานปีใหม่ละแวกนี้ เปิดเสียงเพลงดั...,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\u0E00-\u0E7Fa-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text.lower()

train_df['comment_clean'] = train_df['comment'].apply(preprocess_text)

comparison_df = train_df[['comment', 'comment_clean']].copy()
comparison_df['changed'] = comparison_df['comment'] != comparison_df['comment_clean']
comparison_df[comparison_df['changed']].head()

,comment,comment_clean,changed
12,งานneoncountdown2023 ที่จัดที่สวนสนุก Wonder W...,งานneoncountdown2023 ที่จัดที่สวนสนุก wonder w...,True
14,จัดคอนเสิร์ตใกล้พื้นที่อาศัย แถวนี้มีหมู่บ้านเ...,จัดคอนเสิร์ตใกล้พื้นที่อาศัย แถวนี้มีหมู่บ้านเ...,True
15,ปัญหา: ภายในหมู่บ้านพรไพลิน พบมีการจุดพลุส่งเส...,ปัญหา ภายในหมู่บ้านพรไพลิน พบมีการจุดพลุส่งเสี...,True
17,ปัญหา: ภายในซอย 1 ของหมู่บ้านดังกล่าว พบมีการจ...,ปัญหา ภายในซอย 1 ของหมู่บ้านดังกล่าว พบมีการจุ...,True
19,ปัญหา: ริมถนนดังกล่าว ภายในสวนสนุกวันเดอร์เวิล...,ปัญหา ริมถนนดังกล่าว ภายในสวนสนุกวันเดอร์เวิลด...,True


# Model Selection

In [11]:
random_states = [0, 42, 67]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class_names = list(class_cols)
class2id = {class_name: idx for idx, class_name in enumerate(class_names)}
id2class = {idx: class_name for class_name, idx in class2id.items()}

In [12]:
# Tokenization
def tokenize_dataset(dataset):
    encoded = tokenizer(
        dataset['comment_clean'],
        padding='max_length',
        max_length=128,
        truncation=True,
    )
    return encoded.data

train_dataset = Dataset.from_pandas(train_df[['comment_clean'] + class_names])
train_dataset = train_dataset.rename_columns({col: col for col in class_names})
train_dataset = train_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'] + class_names)
train_dataset

Map: 100%|██████████| 304282/304282 [00:16<00:00, 18711.03 examples/s]


Dataset({
    features: ['comment_clean', 'สำนักงานตำรวจแห่งชาติ', 'การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย', 'สภาเด็กและเยาวชนกรุงเทพมหานคร', 'กรมควบคุมมลพิษ', 'กรมสรรพสามิต', 'การไฟฟ้านครหลวง', 'กรมทางหลวง', 'สำนักงานประกันสุขภาพแห่งชาติ', 'การประปานครหลวง', 'คณะกรรมการการพัฒนาเศรษฐกิจ', 'กระทรวงการท่องเที่ยวและกีฬา', 'สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200', 'input_ids', 'attention_mask'],
    num_rows: 304282
})

In [13]:
def collate_fn(batch):
    return {
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
        'labels': torch.tensor([[float(x[c]) for c in class_names] for x in batch])
    }

# Training

In [14]:
class TraffyClassifier(pl.LightningModule):
    def __init__(self, model_name, num_labels, learning_rate=2e-5, thresh=0.2):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels,
            problem_type="multi_label_classification"
        )
        self.learning_rate = learning_rate
        self.val_preds = []
        self.val_labels = []
        self.thresh = thresh

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self.forward(**batch)
        self.log('train_loss', outputs.loss, prog_bar=True)
        return outputs.loss

    def validation_step(self, batch, batch_idx):
        outputs = self.forward(**batch)
        self.log('val_loss', outputs.loss, prog_bar=True)
        probs = torch.sigmoid(outputs.logits)
        self.val_preds.append(probs.cpu())
        self.val_labels.append(batch['labels'].cpu())

    def on_validation_epoch_end(self):
        preds  = (torch.cat(self.val_preds) > self.thresh).int().numpy()
        labels = torch.cat(self.val_labels).int().numpy()
        f1 = f1_score(labels, preds, average='macro', zero_division=0)
        self.log('val_macro_f1', f1, prog_bar=True)
        self.val_preds = []
        self.val_labels = []

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=0.01)

In [ ]:
trained_models = []
val_loaders = []

for random_state in random_states:
    print(f'=== Training with random_state={random_state} ===')
    
    label_group = train_df['label_group'].values
    train_idx, val_idx = train_test_split(range(len(train_dataset)),
                                          test_size=0.1,
                                          stratify=label_group,
                                          random_state=random_state)

    train_split = train_dataset.select(train_idx)
    val_split   = train_dataset.select(val_idx)

    train_loader = DataLoader(train_split, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_split, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=0)

    pl.seed_everything(random_state)

    # checkpoint_callback = ModelCheckpoint(
    #     monitor='val_macro_f1', mode='max', save_top_k=1,
    #     filename=f'best-rs{random_state}' + '-{epoch:02d}-{val_macro_f1:.4f}'
    # )
    # early_stop = EarlyStopping(monitor='val_macro_f1', patience=2, mode='max')

    # model = TraffyClassifier(model_name=model_name, num_labels=len(class_names))

    # trainer = pl.Trainer(
    #     max_epochs=5,
    #     accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    #     devices=1,
    #     precision='16-mixed',
    #     callbacks=[checkpoint_callback, early_stop]
    # )

    # trainer.fit(model, train_loader, val_loader)
    # print(f'Best model: {checkpoint_callback.best_model_path}')

    # best_model = TraffyClassifier.load_from_checkpoint(
    #     checkpoint_callback.best_model_path,
    #     model_name=model_name, num_labels=len(class_names)
    # )
    # trained_models.append(best_model)
    val_loaders.append(val_loader)

=== Training with random_state=0 ===


Seed set to 0


=== Training with random_state=42 ===


Seed set to 42


=== Training with random_state=67 ===


Seed set to 67


# Testing

In [16]:
test_df = pd.read_csv("data/test.csv")
test_df.head()

,id,comment
0,0,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...
1,1,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...
2,2,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย
3,3,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย
4,4,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...


In [17]:
test_df['comment_clean'] = test_df['comment'].apply(preprocess_text)
test_df.head()

,id,comment,comment_clean
0,0,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...,รถติดจังเลยครับ อยากได้เกาะกลาง ที่ขยับเพิ่มเล...
1,1,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...,ในซอยมีการเตรียมทำท่อระบายน้ำ โดยผู้รับเหมา มา...
2,2,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย,มีต้นไม้กีดขวางทางสัญจรไปมาทำให้เกิดอันตราย
3,3,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย,ร้านนวดบริเวณนี้วางของเกะกะบนทางเท้ามากมาย
4,4,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...


In [18]:
test_dataset = Dataset.from_pandas(test_df[['comment_clean']])
test_dataset = test_dataset.map(tokenize_dataset, batched=True)
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=lambda batch: {
    'input_ids':      torch.stack([x['input_ids'] for x in batch]),
    'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
})

Map: 100%|██████████| 37406/37406 [00:01<00:00, 20437.10 examples/s]


In [19]:
def get_probs(model, loader, device):
    model.eval()
    model.to(device)
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model.model(**batch)
            probs = torch.sigmoid(outputs.logits)
            all_probs.append(probs.cpu())
            if 'labels' in batch:
                all_labels.append(batch['labels'].cpu())
    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).int().numpy() if all_labels else None
    return all_probs, all_labels

# trained_models.append(TraffyClassifier.load_from_checkpoint(
#     "lightning_logs/version_10/checkpoints/best-rs0-epoch=04-val_macro_f1=0.4519.ckpt",
#     model_name=model_name, num_labels=len(class_names))
#                       )

# trained_models.append(TraffyClassifier.load_from_checkpoint(
#     "lightning_logs/version_8/checkpoints/best-rs42-epoch=03-val_macro_f1=0.3665.ckpt",
#     model_name=model_name, num_labels=len(class_names))
#                       )

# trained_models.append(TraffyClassifier.load_from_checkpoint(
#     "lightning_logs/version_9/checkpoints/best-rs67-epoch=03-val_macro_f1=0.3854.ckpt",
#     model_name=model_name, num_labels=len(class_names))
#                       )

# Ensemble val probs
probs_val_list = []
all_labels = None
for i in range(len(trained_models)):
    p, labels = get_probs(trained_models[i], val_loaders[i], device)
    probs_val_list.append(p)
    if all_labels is None:
        all_labels = labels

avg_probs_val = np.mean(probs_val_list, axis=0)

# Per-class threshold
best_thresholds = []
for i in range(len(class_names)):
    best_t, best_f = 0.5, 0
    for t in [0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
        p = (avg_probs_val[:, i] > t).astype(int)
        f = f1_score(all_labels[:, i], p, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    best_thresholds.append(best_t)
    print(f'{class_names[i]}: threshold={best_t:.2f}, f1={best_f:.4f}')

best_thresholds = np.array(best_thresholds)
per_class_f1 = f1_score(all_labels, (avg_probs_val > best_thresholds).astype(int), average='macro', zero_division=0)
print(f'Ensemble Per-class Macro F1: {per_class_f1:.4f}')

# Test inference
probs_test_list = [get_probs(m, test_loader, device)[0] for m in trained_models]
avg_probs_test = np.mean(probs_test_list, axis=0)
all_preds = (avg_probs_test > best_thresholds).astype(int)

submission = pd.DataFrame(all_preds, columns=class_names)
submission.insert(0, 'id', test_df['id'].values)
submission.to_csv('submission.csv', index=False)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 992.68it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
CamembertForSequenceClassification LOAD REPORT from: clicknext/phayathaibert
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkp

สำนักงานตำรวจแห่งชาติ: threshold=0.30, f1=0.4695
การรถไฟฟ้าขนส่งมวลชนแห่งประเทศไทย: threshold=0.30, f1=0.2389
สภาเด็กและเยาวชนกรุงเทพมหานคร: threshold=0.50, f1=0.0000
กรมควบคุมมลพิษ: threshold=0.15, f1=0.2270
กรมสรรพสามิต: threshold=0.10, f1=0.1818
การไฟฟ้านครหลวง: threshold=0.30, f1=0.4066
กรมทางหลวง: threshold=0.25, f1=0.3050
สำนักงานประกันสุขภาพแห่งชาติ: threshold=0.07, f1=0.2105
การประปานครหลวง: threshold=0.30, f1=0.3735
คณะกรรมการการพัฒนาเศรษฐกิจ: threshold=0.10, f1=0.4000
กระทรวงการท่องเที่ยวและกีฬา: threshold=0.07, f1=0.4444
สำนักงาน กสทช. ศูนย์รับแจ้งปัญหา 1200: threshold=0.30, f1=0.3126
Ensemble Per-class Macro F1: 0.2975
